In [9]:
#IMPORTAMOS LAS LIBRERÍAS NECESARIAS
from os import listdir
from numpy import asarray
from numpy import save
import tensorflow as tf
#tf.config.set_visible_devices([], 'GPU')
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPool2D
from tensorflow.keras.layers import Flatten


In [10]:
import tensorflow as tf

# Lista las GPUs disponibles
gpus = tf.config.list_physical_devices('GPU')
print("GPUs detectadas:", gpus)

# Info más detallada
print("Versión de TensorFlow:", tf.__version__)
print("CUDA disponible:", tf.test.is_built_with_cuda())
print("GPU disponible:", tf.test.is_gpu_available())  # deprecated pero útil

GPUs detectadas: []
Versión de TensorFlow: 2.21.0
CUDA disponible: True
GPU disponible: False


W0000 00:00:1777569172.328248   15230 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [11]:
#CARGAMOS EL DATASET QUE RELACIONA LAS FOTOS CON SUS CARACTERÍSTICAS. NUESTRO SISTEMA TIENE QUE APRENDER A PREDECIR ESTOS ATRIBUTOS EN BASE A LAS FOTOS RECIBIDAS.
#SI LA PERSONA ES CALVA O NO, SI ESTÁ SONRIENDO O SI TIENE EL PELO LISO SON CARACTERÍSTICAS DE LAS PERSONAS DE LAS FOTOS QUE EL SISTEMA TIENE QUE APRENDER A 
#PREDECIR.
df = pd.read_csv("CelebrityFaces/list_attr_celeba.csv")
df.replace(-1,0,inplace=True)
df.shape

(202599, 41)

In [12]:
import pandas as pd
df_seleccionado = pd.DataFrame()
gafas = []
sonriendo = []
image_id = []
photos = []
singafas = 0
for idx,row in df.iterrows():
    if row['Eyeglasses']:
        gafas.append(row['Eyeglasses'])
        sonriendo.append(row['Smiling'])
        image_id.append(row['image_id'])
        photo = load_img('CelebrityFaces/img_align_celeba/img_align_celeba/' + row['image_id'], target_size=(50,50), color_mode='grayscale')
        photos.append(img_to_array(photo)/255.)
        del photo
    else:
        if singafas < 13193:
            gafas.append(row['Eyeglasses'])
            sonriendo.append(row['Smiling'])
            image_id.append(row['image_id'])
            singafas += 1
            photo = load_img('CelebrityFaces/img_align_celeba/img_align_celeba/' + row['image_id'], target_size=(50,50), color_mode='grayscale')
            photos.append(img_to_array(photo)/255.)
            del photo
df_seleccionado['Eyeglasses'] = gafas
df_seleccionado['Smiling'] = sonriendo
df_seleccionado['image_id'] = image_id
photos = asarray(photos)
#13193

In [13]:
photos.shape

(26386, 50, 50, 1)

In [14]:
#VAMOS A HACER UN PRIMER INTENTO CON UNA RED NEURONAL NORMAL. COMO SABÉIS NECESITA UNA ENTRADA EN DOS DIMENSIONES. 
#POR ESO METEMOS CAPA FLATTEN.
#HACEMOS UNA PRUEBA COGIENDO SOLO 3 ATRIBUTOS (QUE, EN PRINCIPIO NO TIENEN QUE VER CON EL COLOR DE LAS IMÁGENES)
#HAY QUE ACORDARSE DE LIMITAR LA Y PARA COGER SOLO 100000 FILAS COMO HICIMOS CUANDO COGIMOS LAS FOTOS.
X_train, X_test, y_train, y_test = train_test_split(photos, df_seleccionado[['Eyeglasses','Smiling']], test_size = 0.1, random_state = 0)

In [15]:
#CREAMOS UNA RED NEURONAL NORMAL PARA VER QUE TAL FUNCIONA CON ESTE DATASET. A PRIORI PODRÍA FUNCIONAR BIEN, YA QUE LAS FOTOS ESTÁN BASTANTE CENTRADAS.
model = keras.models.Sequential()
model.add(keras.layers.Flatten(input_shape=[50, 50, 1]))
model.add(keras.layers.Dense(500,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(100,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(2,activation='sigmoid',kernel_initializer='glorot_normal'))

/home/ciabd14/anaconda3/lib/python3.13/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [16]:
#VAMOS A ENTREARLA UN POCO (5 ÉPOCAS)
model.compile(loss='binary_crossentropy', optimizer = keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999), metrics=['binary_accuracy'])
early_stopping_cb = keras.callbacks.EarlyStopping(patience=5,
restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=5,validation_split = 0.1,callbacks=[early_stopping_cb])

Epoch 1/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - binary_accuracy: 0.8037 - loss: 0.4213 - val_binary_accuracy: 0.7541 - val_loss: 0.5828
Epoch 2/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - binary_accuracy: 0.8421 - loss: 0.3575 - val_binary_accuracy: 0.8152 - val_loss: 0.4232
Epoch 3/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - binary_accuracy: 0.8529 - loss: 0.3327 - val_binary_accuracy: 0.8206 - val_loss: 0.3887
Epoch 4/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - binary_accuracy: 0.8595 - loss: 0.3200 - val_binary_accuracy: 0.8653 - val_loss: 0.3171
Epoch 5/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - binary_accuracy: 0.8660 - loss: 0.3092 - val_binary_accuracy: 0.8520 - val_loss: 0.3311


In [17]:
history = model.fit(X_train, y_train, epochs=5,validation_split = 0.1,callbacks=[early_stopping_cb])

Epoch 1/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - binary_accuracy: 0.8647 - loss: 0.3110 - val_binary_accuracy: 0.8366 - val_loss: 0.3730
Epoch 2/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - binary_accuracy: 0.8683 - loss: 0.3037 - val_binary_accuracy: 0.8621 - val_loss: 0.3319
Epoch 3/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - binary_accuracy: 0.8749 - loss: 0.2918 - val_binary_accuracy: 0.8566 - val_loss: 0.3351
Epoch 4/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - binary_accuracy: 0.8797 - loss: 0.2843 - val_binary_accuracy: 0.8651 - val_loss: 0.3249
Epoch 5/5
668/668 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - binary_accuracy: 0.8819 - loss: 0.2769 - val_binary_accuracy: 0.8743 - val_loss: 0.3014


In [21]:
import numpy as np
from sklearn.metrics import accuracy_score

y_pred = (model.predict(X_test) > 0.5).astype(int)
print(accuracy_score(y_test['Eyeglasses'],y_pred[:,0]))

83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 808us/step
0.8715422508525957


In [23]:
model.evaluate(X_test,y_test)


83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - binary_accuracy: 0.8706 - loss: 0.3042  


[0.30424800515174866, 0.8705949187278748]

In [24]:
#VAMOS A PROBAR AHORA CON UNA CONVOLUCIONAL. 

In [33]:
model = keras.models.Sequential([
    # Bloque 1
    Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(50,50,1)),
    Conv2D(32, (3,3), activation='relu', padding='same'),
    MaxPool2D(2,2),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.25),

    # Bloque 2
    Conv2D(64, (3,3), activation='relu', padding='same'),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    MaxPool2D(2,2),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.25),

    # # Bloque 3
    # Conv2D(128, (3,3), activation='relu', padding='same'),
    # MaxPool2D(2,2),
    # keras.layers.BatchNormalization(),
    # keras.layers.Dropout(0.25),

    Flatten(),

    # Cabeza densa — más capacidad que antes
    keras.layers.Dense(128, activation='relu', kernel_initializer='he_normal'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(64, activation='relu', kernel_initializer='he_normal'),

    keras.layers.Dense(2, activation='sigmoid')
])

/home/ciabd14/anaconda3/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [34]:
model.compile(loss='binary_crossentropy', optimizer = keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999), metrics=['accuracy'])
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10,
restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=1000,validation_split = 0.2,callbacks=[early_stopping_cb])

Epoch 1/1000
594/594 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.7063 - loss: 0.4673 - val_accuracy: 0.7084 - val_loss: 0.2695
Epoch 2/1000
594/594 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.7617 - loss: 0.2724 - val_accuracy: 0.8206 - val_loss: 0.2406
Epoch 3/1000
594/594 ━━━━━━━━━━━━━━━━━━━━ 13s 22ms/step - accuracy: 0.7619 - loss: 0.2313 - val_accuracy: 0.8158 - val_loss: 0.2136
Epoch 4/1000
594/594 ━━━━━━━━━━━━━━━━━━━━ 13s 22ms/step - accuracy: 0.7602 - loss: 0.2097 - val_accuracy: 0.7581 - val_loss: 0.1840
Epoch 5/1000
594/594 ━━━━━━━━━━━━━━━━━━━━ 13s 22ms/step - accuracy: 0.7611 - loss: 0.1929 - val_accuracy: 0.7444 - val_loss: 0.1771
Epoch 6/1000
594/594 ━━━━━━━━━━━━━━━━━━━━ 13s 22ms/step - accuracy: 0.7654 - loss: 0.1798 - val_accuracy: 0.7272 - val_loss: 0.2356
Epoch 7/1000
594/594 ━━━━━━━━━━━━━━━━━━━━ 13s 22ms/step - accuracy: 0.7652 - loss: 0.1671 - val_accuracy: 0.7539 - val_loss: 0.1827
Epoch 8/1000
594/594 ━━━━━━━━━━━━━━━━━━━━ 13s 22ms/step - accuracy: 0.7710 -

In [ ]:
y_pred = (model.predict(X_test) > 0.5).astype(int)
print(accuracy_score(y_test['Smiling'],y_pred[:,1]))    # Smiling 1    Eyeglasses 0

83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
0.8973095869647594


In [28]:
y_pred = model.predict(X_test)

83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step


In [29]:
print(y_pred[0:10])

[[9.81301665e-01 1.02397755e-01]
 [9.99996841e-01 1.79132167e-03]
 [7.05529994e-04 9.99925017e-01]
 [1.02764391e-03 3.21460754e-01]
 [8.44137091e-03 7.52697945e-01]
 [8.13359439e-01 9.62789506e-02]
 [8.53697300e-01 9.95836437e-01]
 [9.98145282e-01 2.30711892e-01]
 [6.29566421e-05 9.99897361e-01]
 [1.52423801e-02 4.05401699e-02]]


Este fenómeno es el famoso gradient explosion. 
El modelo empieza a aprender correctamente
Los gradientes se acumulan y se vuelven enormes
Los pesos se actualizan con valores gigantes.
Posibles soluciones:
- Bajar el LR.
- Meter en el optimizador la opción clipnorm=1.0
- Normalizar los datos de entrada (ya lo hemos hecho)
- Meter capas de batchnormalization.

Bajando el LR ya no pasa. Pero no mejora el resultado de validación porque pone todo 0's.


In [30]:
model.evaluate(X_test,y_test)
#SERÍA BUENO PROBAR CON UNA RED MÁS GRANDE TANTO EN CAPAS CONVOLUCIONALES COMO EN NEURONAS DE LA PARTE FULLY CONNECTED.
#SEGURAMENTE OBTENDRÍAMOS UN RESULTADO BASTANTE MEJOR.

83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7579 - loss: 0.1760


[0.17595726251602173, 0.7578628063201904]

In [31]:
y_pred = model.predict(X_test)
for prediccion in y_pred:
    print(prediccion.round())

83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
[1. 0.]
[1. 0.]
[0. 1.]
[0. 0.]
[0. 1.]
[1. 0.]
[1. 1.]
[1. 0.]
[0. 1.]
[0. 0.]
[0. 0.]
[1. 1.]
[1. 0.]
[1. 0.]
[1. 0.]
[0. 1.]
[0. 0.]
[0. 1.]
[1. 0.]
[0. 1.]
[0. 0.]
[0. 0.]
[1. 0.]
[1. 1.]
[1. 1.]
[1. 0.]
[1. 0.]
[0. 0.]
[0. 1.]
[0. 0.]
[0. 0.]
[1. 0.]
[1. 0.]
[0. 0.]
[1. 0.]
[1. 0.]
[0. 0.]
[0. 0.]
[0. 1.]
[0. 0.]
[0. 1.]
[0. 1.]
[1. 1.]
[0. 0.]
[0. 0.]
[1. 0.]
[0. 0.]
[0. 1.]
[1. 1.]
[1. 1.]
[0. 0.]
[1. 1.]
[1. 0.]
[1. 0.]
[0. 0.]
[0. 1.]
[1. 0.]
[0. 0.]
[1. 0.]
[1. 0.]
[1. 0.]
[0. 0.]
[1. 0.]
[1. 1.]
[0. 1.]
[0. 1.]
[1. 0.]
[0. 1.]
[1. 0.]
[1. 1.]
[0. 0.]
[0. 0.]
[1. 1.]
[1. 0.]
[1. 0.]
[0. 0.]
[0. 1.]
[1. 1.]
[0. 0.]
[1. 0.]
[0. 0.]
[1. 1.]
[0. 1.]
[1. 1.]
[0. 0.]
[0. 1.]
[1. 1.]
[1. 0.]
[0. 1.]
[0. 0.]
[0. 1.]
[1. 0.]
[0. 0.]
[0. 1.]
[1. 1.]
[0. 1.]
[0. 1.]
[0. 0.]
[0. 0.]
[0. 1.]
[0. 1.]
[0. 0.]
[0. 0.]
[0. 0.]
[0. 0.]
[1. 0.]
[1. 0.]
[1. 0.]
[0. 1.]
[1. 1.]
[1. 1.]
[1. 0.]
[1. 1.]
[1. 1.]
[0. 1.]
[0. 1.]
[1. 0.]
[0. 0.]
[0. 1.]
[0. 1.]
[